# 중심화·균형 CLV 조건부 구매이력 표현 M2 — Dunnhumby seed 42

직전 분해진단에서 확인된 조건 혼합 붕괴와 가격 관계의 학습 소멸을 직접 수정한 역사적 개발 실험입니다.

- 학습: DAY 1~683, 평가: DAY 684~690 신규상품
- 사용자 조건: N 수준, V 수준, N×V, N−V
- 상품군·가격 구매이력 표현을 유효 고객 평균으로 중심화한 뒤 각각 L2 정규화
- 상품군 비중: `0.25 + 0.5×sigmoid(wᵀx)`, 가격 비중: `1-상품군 비중`
- 두 관계 모두 최소 25%를 유지하여 한 관계가 학습에서 사라지는 것을 차단
- rho는 1~20 epoch 동안 0.005→0.1로 증가하고 이후 0.1 고정
- 하나의 LightGCN·optimizer·plain BPR 안에서 함께 학습
- binary graph, uniform negative sampling, 표본 가중·새 손실 없음
- 최종 test와 holdout을 만들지 않는 seed 42 탐색이며 유의성을 주장하지 않음


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REVIEWED_SHA = '3515631fc93466dc0a19a7cc551ea1d84485af2e'
%cd /content
!rm -rf /content/clv-m2-lightgcn-runner
!git clone -q https://github.com/jung-un/clv-m2-lightgcn-runner.git /content/clv-m2-lightgcn-runner
%cd /content/clv-m2-lightgcn-runner
!git checkout -q $REVIEWED_SHA
import subprocess
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip() == REVIEWED_SHA
print('코드 고정 완료:', REVIEWED_SHA)


In [ ]:
import json
import torch
from lightgcn_clv_conditioned_centered_balanced_history import (
    configure_centered_balanced_history_run,
    preflight_summary,
    run_centered_balanced_history_screen,
)

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
cfg = configure_centered_balanced_history_run(
    out_dir=(
        '/content/drive/MyDrive/논문/data/'
        'results_v3_dunnhumby_m2_clv_conditioned_centered_balanced_history_historical_screen_v1'
    ),
    baseline_result_dir=(
        '/content/drive/MyDrive/논문/data/'
        'results_v3_dunnhumby_m2_repeatshare_historical_backtest_v1'
    ),
)
summary = preflight_summary(cfg)
assert summary['historical_development_split']['final_test_constructed'] is False
assert summary['historical_development_split']['holdout_constructed'] is False
assert summary['m2']['mixer_bounds'] == [0.25, 0.75]
assert summary['m2']['rho_max'] == 0.1
assert summary['m2']['rho_warmup_epochs'] == 20
assert summary['fixed']['graph'] == 'binary'
assert summary['fixed']['negative_sampling'] == 'uniform'
assert summary['fixed']['sample_weighting'] is False
assert summary['fixed']['new_loss_term'] is False
print(json.dumps(summary, ensure_ascii=False, indent=2))


In [ ]:
result_df = run_centered_balanced_history_screen(cfg)


In [ ]:
from IPython.display import display

comparison = result_df.attrs['comparison'].copy()
id_only_comparison = result_df.attrs['id_only_comparison'].copy()
reading = dict(result_df.attrs['screening_reading'])
paths = dict(result_df.attrs['result_paths'])
display_df = result_df.copy()
display_df.attrs = {}
core_metrics = [
    'recall@10', 'ndcg@10', 'recall@20', 'ndcg@20',
    'recall@50', 'ndcg@50',
    'price_purchase_amount_weighted_hit@10',
    'price_purchase_amount_weighted_hit@20',
    'price_purchase_amount_weighted_hit@50',
    'coverage@10', 'n_distinct@10', 'top10_share@10',
]
print('절대지표:')
display(display_df.sort_values('model_id'))
print('M1@64 대비 핵심 변화:')
display(comparison[comparison['metric'].isin(core_metrics)].sort_values('metric'))
print('공동학습 ID-only 대비 full 핵심 변화:')
display(id_only_comparison[id_only_comparison['metric'].isin(core_metrics)].sort_values('metric'))
print('탐색 판독:', reading)
print('결과 파일:', paths)
